# Data Cleaning

## 1. courier waybill info

- dt: Date of the wave.
- courier_id: ID of the rider.
- wave_id: Sequence ID of the work cycle for that specific rider on that day.
- wave_start_time: Time when the wave began (start of a delivery trip).
- wave_end_time: Time when the wave ended (rider becomes idle).
- order_ids: List of Order IDs completed within this wave.

In [4]:
import pandas as pd
courier = pd.read_csv('courier_wave_info_meituan.csv')
courier.dtypes

dt                  int64
courier_id          int64
wave_id             int64
wave_start_time     int64
wave_end_time       int64
order_ids          object
dtype: object

In [5]:
import ast

def clean_wave_data(df):
    # 1. Convert dt
    df['dt'] = pd.to_datetime(df['dt'].astype(str), format='%Y%m%d')
    
    # 2. Convert order_ids from String to Python List
    df['order_ids'] = df['order_ids'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
    
    # 3. Calculate Wave Size (Order count)
    df['wave_size'] = df['order_ids'].apply(len)
    
    # 4. Convert timestamps
    for col in ['wave_start_time', 'wave_end_time']:
        df[col] = pd.to_datetime(df[col], unit='s', errors='coerce')
        
    return df

clean_wave_data(courier)
courier.head()

,dt,courier_id,wave_id,wave_start_time,wave_end_time,order_ids,wave_size
0,2022-10-17,0,1,2022-10-17 13:59:59,2022-10-17 14:20:27,[171478],1
1,2022-10-17,0,2,2022-10-17 15:07:53,2022-10-17 15:33:02,[116902],1
2,2022-10-17,1,1,2022-10-16 16:12:17,2022-10-16 16:29:58,[1],1
3,2022-10-17,1,2,2022-10-17 03:11:57,2022-10-17 04:07:23,"[264834, 107524, 334962, 125682, 493460]",5
4,2022-10-17,2,1,2022-10-16 16:37:57,2022-10-16 17:05:55,"[264776, 2, 398099, 389973]",4


In [6]:
courier.to_csv('courier.csv', index=False, encoding='utf-8-sig')

---
## 2.dispatch ride
- rider_lat / rider_lng: Courier's location at the time of dispatch.
- dispatch_time: Snapshot time of the matching cycle (every 30-60s).
- courier_waybills: Set of order IDs currently carried by the rider (on-hand load).
- courier_id: Candidate courier for potential assignment.

In [8]:
rider = pd.read_csv('dispatch_rider_meituan.csv')
rider.dtypes

Unnamed: 0           int64
dt                   int64
rider_lat            int64
rider_lng            int64
dispatch_time        int64
courier_waybills    object
courier_id           int64
dtype: object

In [9]:
def clean_rider_snapshot(df):
    # 1. Scale coordinates
    df['rider_lat'] = df['rider_lat'] / 1000000.0
    df['rider_lng'] = df['rider_lng'] / 1000000.0
    
    # 2. Parse current on-hand orders and calculate load
    df['courier_waybills'] = df['courier_waybills'].apply(
        lambda x: ast.literal_eval(x) if (isinstance(x, str) and x != '[]') else []
    )
    df['current_load'] = df['courier_waybills'].apply(len)
    
    # 3. Convert dispatch_time
    df['dispatch_time'] = pd.to_datetime(df['dispatch_time'], unit='s', errors='coerce')
    
    return df

clean_rider_snapshot(rider)
rider.head()

,Unnamed: 0,dt,rider_lat,rider_lng,dispatch_time,courier_waybills,courier_id,current_load
0,0,20221017,45.852983,174.574564,2022-10-17 03:25:14,"[284829, 227311]",705,2
1,1,20221017,45.871077,174.595507,2022-10-17 03:25:14,"[165204, 253364, 73375, 483038, 218555, 218582]",392,6
2,2,20221017,45.862480,174.577300,2022-10-17 03:25:14,"[16073, 509042]",3248,2
3,3,20221017,45.912267,174.526266,2022-10-17 03:25:14,"[555809, 310156, 54588, 113651, 28934, 440990,...",993,7
4,4,20221017,45.857203,174.529566,2022-10-17 03:25:14,"[403600, 128710, 506457]",4183,3


In [10]:
rider.to_csv('rider.csv', index=False, encoding='utf-8-sig')

---
## 3.dispatch waybill
- dt: Date of dispatch.
- dispatch_time: Snapshot time of the matching cycle.
- order_id: The ID of the order waiting to be assigned.


In [11]:
waybill = pd.read_csv('dispatch_waybill_meituan.csv')

def clean_dispatch_queue(df):
    # 1. Convert dt and dispatch_time
    df['dt'] = pd.to_datetime(df['dt'].astype(str), format='%Y%m%d')
    df['dispatch_time'] = pd.to_datetime(df['dispatch_time'], unit='s', errors='coerce')
    
    # 2. Check for duplicates in a single dispatch window
    df = df.drop_duplicates(subset=['dispatch_time', 'order_id'])
    
    return df

clean_dispatch_queue(waybill)
waybill.head()

,Unnamed: 0,dt,dispatch_time,order_id
0,0,2022-10-17,2022-10-17 03:25:14,522093
1,1,2022-10-17,2022-10-17 03:25:14,259146
2,2,2022-10-17,2022-10-17 03:25:14,64002
3,3,2022-10-17,2022-10-17 03:25:14,85993
4,4,2022-10-17,2022-10-17 03:25:14,215713


In [12]:
waybill.to_csv('waybill.csv', index=False, encoding='utf-8-sig')

---
## 4. all waybill info
- dt: Date of order creation (converted to datetime).
- order_id / waybill_id: Unique identifiers for the order.
- courier_id: ID of the rider who completed the delivery.
- da_id: Delivery Area ID (note: areas may overlap).
- estimate_arrived_time: Promised delivery deadline for the consumer.
- sender_lng/lat: Location of the merchant (POI).
- recipient_lng/lat: Location of the customer.
- grab_lng/lat: Courier's location at the moment of accepting the order.
- dispatch / grab / fetch / arrive_time: Key lifecycle timestamps (Unix format).
- estimate_meal_prepare_time: Estimated meal ready time (as per Oct 15 supplement).
- platform_order_time: Time when the consumer placed the order.

In [13]:
info = pd.read_csv('all_waybill_info_meituan_0322.csv')

def clean_waybill_data(df):
    # 1. Convert dt to datetime
    df['dt'] = pd.to_datetime(df['dt'].astype(str), format='%Y%m%d')
    
    # 2. Scale coordinates (shifted but not scaled)
    coord_cols = ['sender_lng', 'sender_lat', 'recipient_lng', 'recipient_lat', 'grab_lng', 'grab_lat']
    for col in coord_cols:
        df[col] = df[col] / 1000000.0
        
    # 3. Convert timestamps to Datetime
    time_cols = ['platform_order_time', 'dispatch_time', 'grab_time', 
                 'fetch_time', 'arrive_time', 'estimate_meal_prepare_time', 'estimate_arrived_time']
    for col in time_cols:
        df[col] = pd.to_datetime(df[col], unit='s', errors='coerce')
        
    # 4. Remove canceled/incomplete orders (arrive_time = 0)
    df = df[df['arrive_time'].notnull()].copy()
    
    # 5. Extract Weekday/Weekend features
    df['day_of_week'] = df['dt'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    return df

clean_waybill_data(info)
info.head()

,Unnamed: 0,dt,order_id,waybill_id,courier_id,da_id,is_courier_grabbed,is_weekend,estimate_arrived_time,is_prebook,...,recipient_lat,grab_lng,grab_lat,dispatch_time,grab_time,fetch_time,arrive_time,estimate_meal_prepare_time,order_push_time,platform_order_time
0,0,2022-10-17,0,0,0,0,0,1,2022-10-16 16:32:58,0,...,45.852786,0.000000,0.000000,2022-10-16 16:04:18,1970-01-01 00:00:00,1970-01-01 00:00:00,1970-01-01 00:00:00,2022-10-16 16:12:59,1665936000,2022-10-16 15:59:56
1,1,2022-10-17,1,1,1,1,1,1,2022-10-16 16:31:04,0,...,45.898250,174.530062,45.906005,2022-10-16 16:12:14,2022-10-16 16:12:17,2022-10-16 16:22:24,2022-10-16 16:29:58,2022-10-16 16:14:05,1665936006,2022-10-16 15:59:55
2,2,2022-10-17,2,2,2,2,1,0,2022-10-16 16:58:24,0,...,45.891243,174.548244,45.870923,2022-10-16 16:24:44,2022-10-16 16:25:01,2022-10-16 16:39:06,2022-10-16 16:56:24,2022-10-16 16:33:27,1665937107,2022-10-16 16:18:17
3,3,2022-10-17,3,3,3,0,1,0,2022-10-16 16:57:12,0,...,45.886787,174.560199,45.867948,2022-10-16 16:23:14,2022-10-16 16:23:17,2022-10-16 16:30:30,2022-10-16 17:03:42,2022-10-16 16:27:14,1665937369,2022-10-16 16:17:08
4,4,2022-10-17,4,4,4,0,1,0,2022-10-16 16:48:14,0,...,45.867411,174.554896,45.865167,2022-10-16 16:23:14,2022-10-16 16:23:33,2022-10-16 16:28:30,2022-10-16 16:37:42,2022-10-16 16:28:14,1665937373,2022-10-16 16:18:12


In [14]:
info.to_csv('all_info.csv', index=False, encoding='utf-8-sig')